# Phase 2: Model Quantization and CoT Fine-Tuning (QLoRA)
This notebook implements a hardware-constrained training pipeline for Phi-3 with 4-bit NF4 quantization and LoRA adapters.
It includes dataset inspection, token-length EDA, and training-time diagnostics to support reproducible experiments.

**Note:** This is a refactored, self-contained version with all dependencies inlined - no src/ directory required.

## Environment setup
Install or update the core training stack and visualization tools.

In [ ]:
%pip -q install "transformers>=4.40.0" "peft>=0.10.0" "bitsandbytes>=0.43.0" "accelerate>=0.29.0" "trl>=0.12.0" "datasets>=2.18.0" "tqdm>=4.66.0" "matplotlib>=3.8.0" "seaborn>=0.13.0" "wandb>=0.16.0" "tensorboard>=2.16.0"

## Project root detection
Detect the project root directory.

In [ ]:
# Project Root Detection (vast.ai compatible)
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()  # Use current working directory
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Working directory: {PROJECT_ROOT}")


In [ ]:
# Verify Data Availability
from pathlib import Path

# Check for required data directories
data_dir = PROJECT_ROOT / 'data'
processed_dir = PROJECT_ROOT / 'processed_data'

if not data_dir.exists():
    print(f"❌ Data directory not found: {data_dir}")
    print("   Upload your data to the Vast.ai instance.")
else:
    print(f"✅ Data directory found: {data_dir}")

if not processed_dir.exists():
    print(f"❌ Processed data not found: {processed_dir}")
    print("   Run notebook 01 first to preprocess data.")
else:
    print(f"✅ Processed data found: {processed_dir}")


## Configuration and Constants
Model presets and runtime configuration (inlined from config.py).

In [ ]:
from dataclasses import dataclass
from typing import Dict

# Model presets mapping short keys to Hugging Face model IDs
MODEL_PRESETS: Dict[str, str] = {
    "phi3-mini": "microsoft/Phi-3-mini-4k-instruct",
    "qwen2.5-1.5b": "Qwen/Qwen2.5-1.5B-Instruct",
    "tinyllama-1.1b": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "mistral-7b": "mistralai/Mistral-7B-Instruct-v0.3",
}

@dataclass
class RuntimeConfig:
    """Runtime configuration for training pipeline."""
    model_key: str = "phi3-mini"
    dataset_path: str = "processed_data/finqa_cot_minimal"  # Use cleaned data (minimal cleaning)
    text_field: str = "formatted_prompt"
    output_dir: str = "fingeo_slm_outputs"
    log_dir: str = "fingeo_slm_logs"
    adapter_dir: str = "fingeo-slm-adapter"
    max_train_samples: int = 5000
    max_seq_length: int = 2048
    use_wandb: bool = False
    use_tensorboard: bool = True
    use_qlora: bool = True
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05

## Platform Detection
Detect available hardware backend (CUDA, MPS, or CPU) - inlined from platform.py.

In [ ]:
# GPU Validation and Backend Detection
import torch

def detect_backend():
    """Detect available compute backend."""
    if torch.cuda.is_available():
        return "cuda"
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    else:
        return "cpu"

backend = detect_backend()

if backend != "cuda":
    raise RuntimeError(f"❌ CUDA GPU required! Detected: {backend}. This notebook requires a GPU to run.")

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"✅ GPU Available: {gpu_name}")
print(f"   Memory: {gpu_memory:.1f} GB")
print(f"   Backend: {backend}")
print(f"\n🎯 Training device: {device}")


## Data Loading Functions
Load and validate training data with deduplication - inlined from data.py.

In [ ]:
from typing import Tuple
from datasets import Dataset, DatasetDict, load_from_disk
from pathlib import Path
import os

def load_training_data(
    dataset_path: str,
    text_field: str,
    max_samples: int = 5000,
    split_preference: str = "train",
) -> Tuple[Dataset, Dataset]:
    """Load training data with filtering and deduplication.
    
    Handles both DatasetDict format and split-as-subdirectory format.
    
    Returns:
        Tuple of (full_cleaned_dataset, subset_for_training)
    """
    path = Path(dataset_path)
    
    # Try loading as DatasetDict first
    try:
        dataset = load_from_disk(str(path))
        if isinstance(dataset, DatasetDict):
            if split_preference in dataset:
                train_dataset = dataset[split_preference]
            else:
                first_split = list(dataset.keys())[0]
                print(f"Split '{split_preference}' not found, using '{first_split}'")
                train_dataset = dataset[first_split]
        else:
            train_dataset = dataset
    except Exception:
        # Try loading split as subdirectory
        split_path = path / split_preference
        if split_path.exists():
            print(f"Loading from split directory: {split_path}")
            train_dataset = load_from_disk(str(split_path))
        else:
            # List available splits
            available = [d.name for d in path.iterdir() if d.is_dir() and not d.name.startswith('.')]
            raise FileNotFoundError(
                f"Could not load dataset from {path}. "
                f"Available subdirectories: {available}"
            )

    if text_field not in train_dataset.column_names:
        raise KeyError(f"Expected column '{text_field}' in dataset columns={train_dataset.column_names}")

    cleaned = train_dataset.filter(lambda ex: isinstance(ex[text_field], str) and len(ex[text_field].strip()) > 0)

    # Remove exact duplicate prompts to avoid leakage-like overfitting effects.
    seen = set()
    keep_indices = []
    for idx, text in enumerate(cleaned[text_field]):
        if text not in seen:
            seen.add(text)
            keep_indices.append(idx)
    unique = cleaned.select(keep_indices)

    if len(unique) == 0:
        raise ValueError("No valid training rows after filtering empty and duplicate prompts")

    subset = unique.select(range(min(max_samples, len(unique))))
    return unique, subset


## Model Loading Functions
Tokenizer and model loading with QLoRA support - inlined from modeling.py.

In [ ]:
from typing import Dict, List, Optional
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def resolve_model_id(model_key_or_id: str) -> str:
    """Resolve a model key to its Hugging Face ID."""
    return MODEL_PRESETS.get(model_key_or_id, model_key_or_id)

def load_tokenizer(model_id: str):
    """Load and configure tokenizer."""
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer

def _default_lora_targets(model) -> List[str]:
    """Detect default LoRA target modules."""
    candidates = ["q_proj", "k_proj", "v_proj", "o_proj"]
    names = set()
    for name, _ in model.named_modules():
        for c in candidates:
            if name.endswith(c):
                names.add(c)
    return sorted(names) if names else ["q_proj", "k_proj", "v_proj", "o_proj"]

def _quant_config_for_cuda() -> BitsAndBytesConfig:
    """Create 4-bit quantization config for CUDA."""
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
    )

def load_model_for_training(
    model_id: str,
    backend: str,
    use_qlora: bool,
    lora_r: int,
    lora_alpha: int,
    lora_dropout: float,
) -> Tuple[object, Dict[str, object]]:
    """Load model with optional QLoRA configuration.
    
    Returns:
        Tuple of (model, runtime_info_dict)
    """
    info: Dict[str, object] = {
        "backend": backend,
        "model_id": model_id,
        "qlora_enabled": False,
        "lora_enabled": False,
        "quantization": "none",
        "notes": [],
    }

    if backend == "cuda" and use_qlora:
        quant_config: Optional[BitsAndBytesConfig] = _quant_config_for_cuda()
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=False,
            attn_implementation="eager",
        )
        model = prepare_model_for_kbit_training(model)
        targets = _default_lora_targets(model)
        lora_config = LoraConfig(
            r=lora_r,
            lora_alpha=lora_alpha,
            target_modules=targets,
            lora_dropout=lora_dropout,
            bias="none",
            task_type="CAUSAL_LM",
        )
        model = get_peft_model(model, lora_config)
        info["qlora_enabled"] = True
        info["lora_enabled"] = True
        info["quantization"] = "4bit-nf4"
        info["lora_targets"] = targets
        return model, info

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto" if backend != "cpu" else None,
        trust_remote_code=False,
        attn_implementation="eager",
    )

    if backend == "mps":
        info["notes"].append("QLoRA/4-bit bitsandbytes is CUDA-only. Running full precision on Apple Silicon.")
    elif backend == "cpu":
        info["notes"].append("QLoRA/4-bit bitsandbytes is CUDA-only. Running full precision on CPU.")

    return model, info

## Training Configuration
SFT configuration builder and optimizer selection - inlined from training.py.

In [ ]:
# Training configuration utilities - Enhanced with early stopping
from trl import SFTConfig
from transformers import EarlyStoppingCallback, TrainerCallback
import torch

def choose_optimizer(backend: str, qlora_enabled: bool) -> str:
    """Choose appropriate optimizer based on backend and QLoRA status."""
    if backend == "cuda" and qlora_enabled:
        return "paged_adamw_32bit"
    return "adamw_torch"

def get_gpu_optimized_batch_size(backend: str, vram_gb: float = None) -> tuple:
    """Get batch size and grad accumulation for available GPU."""
    if backend != "cuda":
        return 1, 8
    
    if vram_gb is None:
        try:
            vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
            gpu_name = torch.cuda.get_device_name(0)
            print(f"Detected GPU: {gpu_name} with {vram_gb:.1f}GB VRAM")
        except:
            return 1, 8
    
    if vram_gb >= 80:
        return 8, 2
    elif vram_gb >= 48:
        return 4, 4
    elif vram_gb >= 30:
        return 4, 4
    elif vram_gb >= 24:
        return 2, 8
    elif vram_gb >= 16:
        return 2, 8
    elif vram_gb >= 12:
        return 1, 8
    else:
        return 1, 8

class OverfitMonitorCallback(TrainerCallback):
    """Monitor overfitting by comparing train vs eval loss."""
    
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            eval_loss = metrics.get('eval_loss', None)
            # Get train loss from logs
            train_loss = None
            if state.log_history:
                for log in reversed(state.log_history):
                    if 'loss' in log:
                        train_loss = log['loss']
                        break
            
            if eval_loss and train_loss:
                gap = eval_loss - train_loss
                if gap > 0.3:
                    print(f"\n⚠️  OVERFITTING WARNING: Train loss: {train_loss:.4f}, Eval loss: {eval_loss:.4f} (gap: {gap:.4f})")
                elif gap > 0.15:
                    print(f"\n⚠️  Potential overfitting: Train loss: {train_loss:.4f}, Eval loss: {eval_loss:.4f} (gap: {gap:.4f})")
                else:
                    print(f"✓ Training healthy: Train loss: {train_loss:.4f}, Eval loss: {eval_loss:.4f} (gap: {gap:.4f})")

def build_sft_config(
    output_dir: str,
    log_dir: str,
    report_to: str,
    backend: str,
    qlora_enabled: bool,
    max_seq_length: int,
    num_epochs: int = 3,           # 🔧 Configurable epochs
    enable_early_stopping: bool = True,  # 🔧 Enable early stopping
    early_stopping_patience: int = 2,    # 🔧 Stop if no improvement after N evals
    early_stopping_threshold: float = 0.001,  # 🔧 Minimum improvement threshold
) -> tuple:
    """
    Build SFT training configuration with best model tracking and early stopping.
    
    Returns:
        (SFTConfig, list of callbacks)
    """
    bf16 = backend == "cuda" and torch.cuda.is_bf16_supported()
    fp16 = backend == "cuda" and not bf16
    
    batch_size, grad_accum = get_gpu_optimized_batch_size(backend)
    
    print(f"\n{'='*80}")
    print("TRAINING CONFIGURATION")
    print(f"{'='*80}")
    print(f"  Per-device batch size: {batch_size}")
    print(f"  Gradient accumulation: {grad_accum}")
    print(f"  Effective batch size: {batch_size * grad_accum}")
    print(f"  Precision: {'BF16' if bf16 else 'FP16' if fp16 else 'FP32'}")
    print(f"  Number of epochs: {num_epochs}")
    
    config_params = {
        "output_dir": output_dir,
        "per_device_train_batch_size": batch_size,
        "per_device_eval_batch_size": 1,
        "gradient_accumulation_steps": grad_accum,
        "gradient_checkpointing": True,
        "gradient_checkpointing_kwargs": {"use_reentrant": False},
        "optim": choose_optimizer(backend, qlora_enabled),
        "learning_rate": 2e-4,
        "logging_steps": 10,
        "num_train_epochs": num_epochs,  # Use configurable epochs
        "bf16": bf16,
        "fp16": fp16,
        "report_to": report_to,
        "logging_dir": log_dir,
        
        # 🎯 BEST MODEL TRACKING - Prevents overfitting
        "save_strategy": "epoch",              # Save after each epoch
        "eval_strategy": "epoch",              # Evaluate after each epoch
        "save_total_limit": 3,                 # Keep best + 2 recent checkpoints
        "load_best_model_at_end": True,        # ✓ Load best model (not last!)
        "metric_for_best_model": "eval_loss",  # Use validation loss as metric
        "greater_is_better": False,            # Lower loss = better
        
        # Regularization
        "max_grad_norm": 1.0,                  # Gradient clipping
        "warmup_steps": 10,
        "weight_decay": 0.01,                  # L2 regularization
    }
    
    try:
        config = SFTConfig(
            **config_params,
            max_seq_length=max_seq_length,
            dataset_text_field="formatted_prompt",
            packing=False,
        )
    except TypeError:
        try:
            config = SFTConfig(
                **config_params,
                dataset_text_field="formatted_prompt",
                packing=False,
                dataset_kwargs={"max_length": max_seq_length},
            )
        except TypeError:
            config = SFTConfig(
                **config_params,
                dataset_text_field="formatted_prompt",
                packing=False,
            )
    
    # Setup callbacks
    callbacks = []
    
    # Overfitting monitor (always enabled)
    callbacks.append(OverfitMonitorCallback())
    
    # Early stopping (optional)
    if enable_early_stopping:
        early_stop = EarlyStoppingCallback(
            early_stopping_patience=early_stopping_patience,
            early_stopping_threshold=early_stopping_threshold
        )
        callbacks.append(early_stop)
        print(f"\n{'='*80}")
        print("OVERFITTING PREVENTION")
        print(f"{'='*80}")
        print(f"  ✓ Best Model Tracking: ENABLED")
        print(f"    - Saves checkpoint after each epoch")
        print(f"    - Evaluates on validation set")
        print(f"    - Keeps only best model based on eval_loss")
        print(f"  ✓ Early Stopping: ENABLED")
        print(f"    - Patience: {early_stopping_patience} epochs")
        print(f"    - Threshold: {early_stopping_threshold}")
        print(f"    - Stops if eval_loss doesn't improve")
        print(f"  ✓ Overfitting Monitor: ENABLED")
        print(f"    - Tracks train vs eval loss gap")
        print(f"    - Warns if model is overfitting")
    else:
        print(f"\n  ✓ Best Model Tracking: ENABLED")
        print(f"    - Evaluates after each epoch")
        print(f"    - Loads best checkpoint at end")
        print(f"  ⚠️  Early Stopping: DISABLED")
    
    return config, callbacks

print("✓ Enhanced training config with overfitting prevention loaded")


## Imports and Global Configuration
Seed control and lightweight helper utilities used throughout the notebook.

In [ ]:
import textwrap
from typing import List
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import TrainerCallback, set_seed
from trl import SFTTrainer

set_seed(42)
sns.set_theme(style="whitegrid")

def log_gpu_memory(tag: str) -> None:
    """Log GPU memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024 ** 3)
        reserved = torch.cuda.memory_reserved() / (1024 ** 3)
        print(f"[GPU] {tag} | allocated={allocated:.2f} GB, reserved={reserved:.2f} GB")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        allocated = torch.mps.current_allocated_memory() / (1024 ** 3)
        print(f"[MPS] {tag} | allocated={allocated:.2f} GB")
    else:
        print(f"[MEM] {tag} | accelerator memory metrics not available")

runtime = RuntimeConfig()
print("Available model presets:")
for k, v in MODEL_PRESETS.items():
    print(f"- {k}: {v}")
print(f"\nCurrent model preset: {runtime.model_key}")


## Load Processed Dataset
Validate the dataset schema and select a training subset for rapid iteration.

In [ ]:
print("Loading and validating training dataset...")

try:
    train_dataset, train_data = load_training_data(
        dataset_path=runtime.dataset_path,
        text_field=runtime.text_field,
        max_samples=runtime.max_train_samples,
        split_preference="train",
    )
except Exception as exc:
    raise RuntimeError(f"Failed to load/validate dataset from {runtime.dataset_path}") from exc

print(f"Total clean examples: {len(train_dataset)} | Using subset: {len(train_data)}")
print(f"Columns: {train_dataset.column_names}")

if len(train_dataset) != len(train_data):
    print(f"Subset mode is active for faster iteration: {len(train_data)} / {len(train_dataset)}")

## Dataset Statistics Visualization
Visualize dataset composition and field statistics.

In [ ]:
# Visualize dataset size comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Dataset size bar chart
axes[0].bar(['Full Dataset', 'Training Subset'], [len(train_dataset), len(train_data)], color=['#2a9d8f', '#e76f51'])
axes[0].set_ylabel('Number of Examples')
axes[0].set_title('Dataset Size Comparison')
axes[0].grid(axis='y', alpha=0.3)

# Column count
axes[1].barh(['Dataset'], [len(train_dataset.column_names)], color='#264653')
axes[1].set_xlabel('Number of Columns')
axes[1].set_title('Dataset Schema')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nDataset columns: {', '.join(train_dataset.column_names)}")

## Tokenizer Setup and Prompt Inspection
Load the tokenizer and print a formatted example CoT prompt.

In [ ]:
# To switch SLMs, change runtime.model_key to a preset key or direct Hugging Face model id.
# Example: runtime.model_key = "qwen2.5-1.5b" or runtime.model_key = "mistralai/Mistral-7B-Instruct-v0.3"
model_id = resolve_model_id(runtime.model_key)
print(f"Selected model id: {model_id}")

print("Loading tokenizer...")
tokenizer = load_tokenizer(model_id)

sample_prompt = train_data[0][runtime.text_field]
print("Sample formatted_prompt:\n")
print(textwrap.fill(sample_prompt, width=100))

## EDA: Token Length Distribution
Compute token lengths with a progress bar and visualize the distribution.

In [ ]:
eda_sample = train_data.select(range(min(2000, len(train_data))))

def add_token_length(batch):
    tokenized = tokenizer(batch["formatted_prompt"], add_special_tokens=False)
    return {"token_length": [len(ids) for ids in tokenized["input_ids"]]}

try:
    eda_with_len = eda_sample.map(add_token_length, batched=True, desc="Computing token lengths")
    lengths = eda_with_len["token_length"]
    print(
        "Token length stats -> "
        f"min: {min(lengths)} | max: {max(lengths)} | mean: {np.mean(lengths):.2f} | median: {np.median(lengths):.2f}"
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Histogram with KDE
    axes[0].hist(lengths, bins=40, color="#2a9d8f", alpha=0.7, edgecolor='black')
    axes[0].axvline(np.mean(lengths), color='#e76f51', linestyle='--', linewidth=2, label=f'Mean: {np.mean(lengths):.0f}')
    axes[0].axvline(np.median(lengths), color='#264653', linestyle='--', linewidth=2, label=f'Median: {np.median(lengths):.0f}')
    axes[0].set_title("Token Length Distribution")
    axes[0].set_xlabel("Token length")
    axes[0].set_ylabel("Count")
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    # Box plot
    axes[1].boxplot(lengths, vert=True, patch_artist=True,
                    boxprops=dict(facecolor='#2a9d8f', alpha=0.7),
                    medianprops=dict(color='#e76f51', linewidth=2))
    axes[1].set_title("Token Length Box Plot")
    axes[1].set_ylabel("Token length")
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print percentiles
    print(f"\nPercentiles:")
    for p in [25, 50, 75, 90, 95, 99]:
        print(f"  {p}th: {np.percentile(lengths, p):.0f}")
        
except Exception as exc:
    print(f"EDA failed: {exc}")

## Quantization Config and Model Load
Load model with 4-bit NF4 and prepare it for QLoRA training.

> **Macbook / Apple Silicon note (important):**
>
> `bitsandbytes` 4-bit QLoRA kernels are CUDA-only. On MPS, this notebook automatically falls back to full precision training and `adamw_torch` optimizer.
> You can still train or debug on Mac, but true QLoRA benchmarking requires a CUDA GPU (NVIDIA).

> **TPU note:**
>
> TPU does not support `bitsandbytes` 4-bit quantization. QLoRA will be automatically disabled on TPU, and training will use mixed-precision FP16/BF16 instead. For QLoRA training, use **NVIDIA GPU (CUDA)** instead of TPU.

In [ ]:
backend = detect_backend()
print(f"Detected backend: {backend}")

print("Loading model with backend-aware strategy...")
try:
    model, model_runtime_info = load_model_for_training(
        model_id=model_id,
        backend=backend,
        use_qlora=runtime.use_qlora,
        lora_r=runtime.lora_r,
        lora_alpha=runtime.lora_alpha,
        lora_dropout=runtime.lora_dropout,
    )
except Exception as exc:
    raise RuntimeError(
        "Model loading failed. Check model id, internet/auth access, and backend compatibility."
    ) from exc

print("Model runtime info:")
for k, v in model_runtime_info.items():
    print(f"- {k}: {v}")

## Model Parameter Analysis
Count trainable parameters and visualize model configuration.

In [ ]:
def count_trainable_params(model_obj) -> Tuple[int, int, float]:
    """Count trainable and total parameters."""
    trainable = sum(p.numel() for p in model_obj.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model_obj.parameters())
    pct = 100 * trainable / max(1, total)
    print(f"Trainable params: {trainable:,} / {total:,} ({pct:.2f}%)")
    return trainable, total, pct

model_for_training = model
trainable, total, pct = count_trainable_params(model_for_training)

if model_runtime_info.get("qlora_enabled", False):
    print("QLoRA mode is active (4-bit + LoRA adapters).")
else:
    print("Full precision mode is active (no 4-bit QLoRA adapters on this backend).")

# Visualize parameter breakdown
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
axes[0].pie([trainable, total - trainable], 
            labels=['Trainable', 'Frozen'],
            colors=['#2a9d8f', '#e9c46a'],
            autopct='%1.1f%%',
            startangle=90)
axes[0].set_title('Parameter Distribution')

# Bar chart
axes[1].bar(['Trainable', 'Frozen'], 
            [trainable / 1e6, (total - trainable) / 1e6],
            color=['#2a9d8f', '#e9c46a'])
axes[1].set_ylabel('Parameters (Millions)')
axes[1].set_title('Parameter Count by Type')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---

**Device-aware LoRA/QLoRA logic:**
- On CUDA: LoRA/QLoRA adapters and 4-bit quantization are enabled for efficient training.
- On MPS/CPU: LoRA/QLoRA and quantization are skipped. Training proceeds in full precision. This is slower and more memory-intensive, but avoids unsupported operation errors.

---

## Training/Validation Split
Create an internal validation split from cleaned subset.

In [ ]:
# Thesis-grade split discipline: create an internal validation split from cleaned subset.
split = train_data.train_test_split(test_size=0.05, seed=42)
train_split_data = split["train"]
eval_split_data = split["test"]

print(f"Train split: {len(train_split_data)} | Eval split: {len(eval_split_data)}")
print("Using cleaned, deduplicated prompts to reduce data leakage and duplicated supervision.")

# Visualize split
plt.figure(figsize=(8, 4))
plt.bar(['Training', 'Validation'], [len(train_split_data), len(eval_split_data)], 
        color=['#2a9d8f', '#e76f51'])
plt.ylabel('Number of Examples')
plt.title('Train/Validation Split')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Training Configuration and Logging
Configure TRL SFT settings and optional experiment tracking.

In [ ]:
use_wandb = runtime.use_wandb
use_tensorboard = runtime.use_tensorboard

# Create output and log directories upfront to avoid assertion errors
os.makedirs(runtime.output_dir, exist_ok=True)
os.makedirs(runtime.log_dir, exist_ok=True)

if use_wandb:
    import wandb

    wandb.init(project="fingeo-slm", name=f"{runtime.model_key}-training")

report_to = "wandb" if use_wandb else ("tensorboard" if use_tensorboard else "none")

# 🔧 CONFIGURE TRAINING HERE
NUM_EPOCHS = 10  # Change to 5, 10, etc. for more training
ENABLE_EARLY_STOPPING = True  # Set False to train all epochs
EARLY_STOPPING_PATIENCE = 2  # Stop after 2 epochs without improvement

sft_config, training_callbacks = build_sft_config(
    output_dir=runtime.output_dir,
    log_dir=runtime.log_dir,
    report_to=report_to,
    backend=backend,
    qlora_enabled=bool(model_runtime_info.get("qlora_enabled", False)),
    max_seq_length=runtime.max_seq_length,
    num_epochs=NUM_EPOCHS,
    enable_early_stopping=ENABLE_EARLY_STOPPING,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
)

print("Training configuration summary:")
print(f"- backend: {backend}")
print(f"- optimizer: {sft_config.optim}")
print(f"- bf16: {sft_config.bf16} | fp16: {sft_config.fp16}")
print(f"- report_to: {report_to}")
print(f"- output_dir: {runtime.output_dir}")
print(f"- log_dir: {runtime.log_dir}")
print(f"- learning_rate: {sft_config.learning_rate}")
print(f"- per_device_batch_size: {sft_config.per_device_train_batch_size}")
print(f"- gradient_accumulation_steps: {sft_config.gradient_accumulation_steps}")
print(f"- num_train_epochs: {sft_config.num_train_epochs}")

## Training Loop with Diagnostics
Track loss, profile GPU memory, and persist the model.

In [ ]:
# ================================================================================
# Memory Management Setup (Run Before Training)
# ================================================================================
import os
import gc

# Set PyTorch memory allocation configuration
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Clear any cached memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    print("✓ CUDA cache cleared")
    print(f"✓ PYTORCH_CUDA_ALLOC_CONF set to: {os.environ['PYTORCH_CUDA_ALLOC_CONF']}")
    
    # Display current memory status
    allocated = torch.cuda.memory_allocated() / (1024**3)
    reserved = torch.cuda.memory_reserved() / (1024**3)
    total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    free = total - allocated
    
    print(f"\n📊 GPU Memory Status Before Training:")
    print(f"  Total VRAM: {total:.2f} GB")
    print(f"  Allocated: {allocated:.2f} GB ({allocated/total*100:.1f}%)")
    print(f"  Reserved: {reserved:.2f} GB ({reserved/total*100:.1f}%)")
    print(f"  Free: {free:.2f} GB ({free/total*100:.1f}%)")
    
    if allocated > total * 0.5:
        print("\n⚠️  WARNING: More than 50% VRAM already allocated!")
        print("   Consider restarting the kernel to free memory.")
    else:
        print("\n✅ Memory looks good - ready for training")

print("\n💡 Memory Optimization Tips:")
print("  - Current batch_size: {}".format(sft_config.per_device_train_batch_size))
print("  - Gradient accumulation: {}".format(sft_config.gradient_accumulation_steps))
print("  - Effective batch size: {}".format(sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps))
print("  - Packing enabled: {}".format(sft_config.packing))
print("  - Gradient checkpointing: {}".format(sft_config.gradient_checkpointing))


In [ ]:
# Training with Epoch Tracking
from typing import List, Tuple

loss_history: List[Tuple[int, float]] = []
eval_history: List[Tuple[int, float]] = []
epoch_metrics: List[dict] = []

class EpochTrackingCallback(TrainerCallback):
    """Track training and evaluation metrics per epoch."""
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            loss_history.append((state.global_step, float(logs["loss"])))
    
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            eval_loss = metrics.get("eval_loss", None)
            if eval_loss is not None:
                eval_history.append((state.epoch, eval_loss))
                epoch_metrics.append({
                    "epoch": state.epoch,
                    "eval_loss": eval_loss,
                    "step": state.global_step
                })
                print(f"\n📊 Epoch {state.epoch:.0f} | Eval Loss: {eval_loss:.4f}")

trainer = SFTTrainer(
    model=model_for_training,
    train_dataset=train_split_data,
    eval_dataset=eval_split_data,
    args=sft_config,
    processing_class=tokenizer,
    callbacks=[EpochTrackingCallback()],
)

training_succeeded = False
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
log_gpu_memory("Before training")

try:
    print("Starting CoT fine-tuning with epoch tracking...")
    print("="*60)
    trainer.train()
    training_succeeded = True
except Exception as exc:
    print(f"Training failed: {exc}")
    import traceback
    traceback.print_exc()
finally:
    log_gpu_memory("After training")
    if torch.cuda.is_available():
        peak = torch.cuda.max_memory_allocated() / (1024 ** 3)
        print(f"[GPU] Peak allocated={peak:.2f} GB")

# Show best epoch summary
if epoch_metrics:
    print("\n" + "="*60)
    print("EPOCH SUMMARY")
    print("="*60)
    best_epoch = min(epoch_metrics, key=lambda x: x["eval_loss"])
    
    print(f"\n{'Epoch':<10} {'Eval Loss':<15} {'Status':<10}")
    print("-"*35)
    for em in epoch_metrics:
        status = "⭐ BEST" if em["epoch"] == best_epoch["epoch"] else ""
        print(f"{em['epoch']:<10.0f} {em['eval_loss']:<15.4f} {status}")
    
    print(f"\n✓ Best model from Epoch {best_epoch['epoch']:.0f} (loss: {best_epoch['eval_loss']:.4f})")
    print("  This model will be automatically loaded and saved.")

print("\n" + "="*80)
print("SAVING MODEL")
print("="*80)

if training_succeeded:
    # Save the adapter (LoRA weights)
    adapter_dir = PROJECT_ROOT / runtime.output_dir / runtime.adapter_dir
    adapter_dir.mkdir(parents=True, exist_ok=True)
    
    if model_runtime_info.get("lora_enabled", False):
        print(f"\nSaving LoRA adapter to: {adapter_dir}")
        trainer.model.save_pretrained(adapter_dir)
        tokenizer.save_pretrained(adapter_dir)
        print(f"✓ LoRA adapter saved to: {adapter_dir}")
    else:
        print(f"\nSaving full model to: {adapter_dir}")
        trainer.model.save_pretrained(adapter_dir)
        tokenizer.save_pretrained(adapter_dir)
        print(f"✓ Full model saved to: {adapter_dir}")
    
    # Save merged model for deployment
    final_model_dir = PROJECT_ROOT / runtime.output_dir / "finetuned_model"
    final_model_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\nCreating merged model for deployment...")
    try:
        if model_runtime_info.get("lora_enabled", False):
            merged_model = trainer.model.merge_and_unload()
            merged_model.save_pretrained(final_model_dir)
            tokenizer.save_pretrained(final_model_dir)
            print(f"✓ Merged model saved to: {final_model_dir}")
        else:
            trainer.model.save_pretrained(final_model_dir)
            tokenizer.save_pretrained(final_model_dir)
            print(f"✓ Model saved to: {final_model_dir}")
    except Exception as e:
        print(f"⚠️ Could not save merged model: {e}")
        print("  LoRA adapter is still available at:", adapter_dir)
else:
    print("⚠️ Training did not complete - model not saved")


## Training Loss Visualization
Plot the loss curve collected by the callback.

In [ ]:
# Training and Evaluation Visualization
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Training Progress and Best Epoch Analysis', fontsize=14, fontweight='bold')

# 1. Training Loss Over Steps
ax1 = axes[0, 0]
if loss_history:
    steps, losses = zip(*loss_history)
    ax1.plot(steps, losses, color='#2a9d8f', linewidth=1.5, alpha=0.7)
    ax1.scatter(steps[::len(steps)//10+1], losses[::len(steps)//10+1], 
                color='#2a9d8f', s=20, zorder=5)
    ax1.set_xlabel('Training Steps')
    ax1.set_ylabel('Training Loss')
    ax1.set_title('Training Loss Over Steps')
    ax1.grid(True, alpha=0.3)
else:
    ax1.text(0.5, 0.5, 'No training data', ha='center', va='center')
    ax1.set_title('Training Loss (No Data)')

# 2. Evaluation Loss Per Epoch - BEST EPOCH HIGHLIGHT
ax2 = axes[0, 1]
if epoch_metrics:
    epochs = [em['epoch'] for em in epoch_metrics]
    eval_losses = [em['eval_loss'] for em in epoch_metrics]
    best_idx = np.argmin(eval_losses)
    
    # Bar chart with best epoch highlighted
    colors = ['#e76f51' if i != best_idx else '#2a9d8f' for i in range(len(epochs))]
    bars = ax2.bar(epochs, eval_losses, color=colors, edgecolor='black', linewidth=1.5)
    
    # Add value labels
    for bar, loss in zip(bars, eval_losses):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{loss:.4f}', ha='center', va='bottom', fontsize=10)
    
    # Mark best epoch
    ax2.axhline(y=eval_losses[best_idx], color='#2a9d8f', linestyle='--', alpha=0.5)
    ax2.annotate(f'Best: Epoch {epochs[best_idx]:.0f}', 
                xy=(epochs[best_idx], eval_losses[best_idx]),
                xytext=(epochs[best_idx] + 0.3, eval_losses[best_idx] + 0.05),
                fontsize=11, fontweight='bold', color='#2a9d8f',
                arrowprops=dict(arrowstyle='->', color='#2a9d8f'))
    
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Evaluation Loss')
    ax2.set_title('Evaluation Loss Per Epoch (⭐ = Best)')
    ax2.set_xticks(epochs)
    ax2.grid(True, alpha=0.3, axis='y')
else:
    ax2.text(0.5, 0.5, 'No evaluation data', ha='center', va='center')
    ax2.set_title('Evaluation Loss (No Data)')

# 3. Training Loss Smoothed (Moving Average)
ax3 = axes[1, 0]
if loss_history and len(loss_history) > 10:
    steps, losses = zip(*loss_history)
    window = min(50, len(losses) // 5)
    if window > 1:
        smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
        smooth_steps = steps[window-1:]
        ax3.plot(smooth_steps, smoothed, color='#264653', linewidth=2, label='Smoothed Loss')
        ax3.fill_between(smooth_steps, smoothed, alpha=0.3, color='#264653')
    ax3.plot(steps, losses, color='#2a9d8f', alpha=0.3, linewidth=0.5, label='Raw Loss')
    ax3.set_xlabel('Training Steps')
    ax3.set_ylabel('Loss')
    ax3.set_title('Training Loss (Smoothed)')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
else:
    ax3.text(0.5, 0.5, 'Not enough data', ha='center', va='center')

# 4. Summary Table
ax4 = axes[1, 1]
ax4.axis('off')

if epoch_metrics:
    best = min(epoch_metrics, key=lambda x: x['eval_loss'])
    worst = max(epoch_metrics, key=lambda x: x['eval_loss'])
    
    summary_text = f"""
    ╔════════════════════════════════════════╗
    ║       TRAINING SUMMARY                 ║
    ╠════════════════════════════════════════╣
    ║  Total Epochs:     {len(epoch_metrics):<20}║
    ║  Total Steps:      {loss_history[-1][0] if loss_history else 0:<20}║
    ╠════════════════════════════════════════╣
    ║  ⭐ BEST EPOCH:    {best['epoch']:.0f}                   ║
    ║     Eval Loss:     {best['eval_loss']:.4f}              ║
    ╠════════════════════════════════════════╣
    ║  Worst Epoch:      {worst['epoch']:.0f}                   ║
    ║     Eval Loss:     {worst['eval_loss']:.4f}              ║
    ╠════════════════════════════════════════╣
    ║  Improvement:      {((worst['eval_loss'] - best['eval_loss']) / worst['eval_loss'] * 100):.1f}%               ║
    ╚════════════════════════════════════════╝
    
    The model from Epoch {best['epoch']:.0f} has been saved.
    """
    ax4.text(0.1, 0.5, summary_text, fontfamily='monospace', fontsize=11,
             verticalalignment='center', bbox=dict(boxstyle='round', facecolor='#f0f0f0'))
else:
    ax4.text(0.5, 0.5, 'Training summary not available', ha='center', va='center')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / runtime.output_dir / 'training_epochs_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Visualization saved to: {PROJECT_ROOT / runtime.output_dir / 'training_epochs_analysis.png'}")


## Evaluation Metrics Visualization
If evaluation was performed, visualize evaluation metrics.

In [ ]:
if "trainer" in globals() and hasattr(trainer, 'state') and trainer.state.log_history:
    eval_logs = [log for log in trainer.state.log_history if 'eval_loss' in log]
    
    if eval_logs:
        eval_steps = [log.get('step', i) for i, log in enumerate(eval_logs)]
        eval_losses = [log['eval_loss'] for log in eval_logs]
        
        plt.figure(figsize=(8, 4))
        plt.plot(eval_steps, eval_losses, marker='s', color='#e76f51', linewidth=2, markersize=6, label='Eval Loss')
        plt.title('Evaluation Loss During Training')
        plt.xlabel('Step')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        print(f"Best eval loss: {min(eval_losses):.4f} at step {eval_steps[eval_losses.index(min(eval_losses))]}")
    else:
        print("No evaluation metrics found in training logs.")
else:
    print("Trainer not available or training not completed.")

## End-of-Notebook Validation
Verify that key training artifacts and logs exist after execution.

In [ ]:
# Validation: Check that training pipeline completed successfully
import os
from pathlib import Path

assert isinstance(train_data, Dataset), "train_data is not a Dataset instance"
assert len(train_data) > 0, "train_data is empty"
assert len(train_split_data) > 0, "train split is empty"
assert len(eval_split_data) > 0, "eval split is empty"
assert os.path.exists(f"./{runtime.output_dir}"), "Output directory missing"
assert os.path.exists(f"./{runtime.log_dir}"), "Log directory missing"

if training_succeeded:
    # Check for saved models at correct locations (updated paths)
    adapter_path = PROJECT_ROOT / runtime.output_dir / runtime.adapter_dir
    final_model_path = PROJECT_ROOT / runtime.output_dir / "finetuned_model"
    
    saved_adapter = adapter_path.exists()
    saved_final = final_model_path.exists()
    
    print(f"\nModel save validation:")
    print(f"  Adapter path: {adapter_path}")
    print(f"  Adapter exists: {saved_adapter}")
    print(f"  Final model path: {final_model_path}")
    print(f"  Final model exists: {saved_final}")
    
    if saved_adapter:
        adapter_files = list(adapter_path.glob("*"))
        print(f"  Adapter files: {len(adapter_files)} files")
    
    if saved_final:
        final_files = list(final_model_path.glob("*"))
        print(f"  Final model files: {len(final_files)} files")
        
        # Check for critical model files
        has_config = (final_model_path / "config.json").exists()
        has_weights = (final_model_path / "model.safetensors").exists() or \
                      (final_model_path / "pytorch_model.bin").exists()
        has_tokenizer = (final_model_path / "tokenizer.json").exists() or \
                        (final_model_path / "tokenizer_config.json").exists()
        
        print(f"\n  Model completeness check:")
        print(f"    Config: {'✓' if has_config else '✗'}")
        print(f"    Weights: {'✓' if has_weights else '✗'}")
        print(f"    Tokenizer: {'✓' if has_tokenizer else '✗'}")
        
        if has_config and has_weights and has_tokenizer:
            print(f"\n  ✅ Model is complete and ready to use!")
        else:
            print(f"\n  ⚠️ Model may be incomplete")
    
    # Assert at least one model was saved
    assert saved_adapter or saved_final, \
        f"Expected trained model missing! Checked:\n  - {adapter_path}\n  - {final_model_path}"
else:
    print("\n⚠️ Training did not complete successfully - skipping model validation")

print("\n" + "="*80)
print("VALIDATION COMPLETE")
print("="*80)
print(f"\nSummary:")
print(f"  Backend: {backend}")
print(f"  Model: {model_id}")
print(f"  Training samples: {len(train_split_data)}")
print(f"  Validation samples: {len(eval_split_data)}")
print(f"  Training succeeded: {training_succeeded}")
if training_succeeded:
    print(f"  Models saved: {'Adapter' if saved_adapter else ''}{' + ' if saved_adapter and saved_final else ''}{'Final' if saved_final else 'None'}")
    print(f"\n✅ Phase 2 notebook executed end-to-end successfully!")
    print(f"\n📦 Your model is ready at:")
    if saved_final:
        print(f"   {final_model_path}")
    elif saved_adapter:
        print(f"   {adapter_path}")


In [ ]:
# Model Saved Locally
print("\n" + "="*70)
print("✅ MODEL TRAINING COMPLETE")
print("="*70)

final_model_path = PROJECT_ROOT / runtime.output_dir / "finetuned_model"
adapter_path = PROJECT_ROOT / runtime.output_dir / runtime.adapter_dir

print(f"\nFine-tuned model saved to:")
print(f"  📁 {final_model_path}")
print(f"\nLoRA adapter saved to:")
print(f"  📁 {adapter_path}")

print(f"\n💾 To download from Vast.ai:")
print(f"  tar -czvf model.tar.gz {runtime.output_dir}")
print(f"  # Then use Vast.ai file download or scp")

print(f"\n📊 Next steps:")
print(f"  1. Run Notebook 03 for evaluation & benchmarking")
print(f"  2. Run Notebook 05 for logical reasoning tests")
print(f"  3. Compare with baseline performance")
